# MODULE 35.3 — SPECTRAL FEATURES FOR EMOTION RECOGNITION 🎧

## 1. Why Do We Need Spectral Features?

In Module 35.2, we extracted time-domain characteristics such as RMS and Zero-Crossing Rate (ZCR).

Now we'll investigate how the frequency content of a voice changes across different emotions.

A waveform tells us how the signal's amplitude changes over time.

But speech also contains frequency information.

Consider two people saying the same sentence:

- Person A speaks softly.
- Person B speaks with a sharp, forceful voice.

Their waveforms may have similar durations, but their frequency distributions can differ.

Spectral features help describe characteristics such as:

- Where the signal's frequency content is concentrated.
- How widely that content is distributed.
- How much energy exists in different frequency regions.
- How quickly the frequency distribution changes.

**Our goal:** Convert spectral information into numerical features that a machine-learning model can use.

---

## 2. Spectral Centroid

<badge color="success">Core Feature</badge>

### What is it?

Spectral centroid represents the centre of mass of a magnitude spectrum.

In simple terms:

It tells us where the spectral energy is concentrated along the frequency axis.

Imagine two audio frames:

- Frame A has most of its energy concentrated at lower frequencies.
- Frame B has more energy concentrated at higher frequencies.

Frame B will generally have a higher spectral centroid.

### Mathematical Representation

$$
C_t =
\frac{
\sum_{k=0}^{K-1} f_k |X_t[k]|
}{
\sum_{k=0}^{K-1}|X_t[k]|
}
$$

Where:

- \(C_t\) = Spectral centroid of frame \(t\).
- \(f_k\) = Frequency of FFT bin \(k\).
- \(X_t[k]\) = Complex spectrum at bin \(k\).
- \(|X_t[k]|\) = Magnitude of that frequency component.

### Python Implementation

```python
centroid = librosa.feature.spectral_centroid(
    y=audio,
    sr=TARGET_SR,
    n_fft=512,
    hop_length=160,
    win_length=400
)[0]
```

The output is a sequence of centroid values, usually expressed in Hz.

### What Can It Tell Us About Emotion?

A brighter or sharper-sounding voice may have a higher spectral centroid.

However, a higher centroid does not automatically mean anger or happiness.

It is also affected by:

- Speaker characteristics.
- Vocal pitch and articulation.
- Microphone characteristics.
- Background noise.
- Recording conditions.

**SER application:** Useful as one component of a larger feature representation.

---

## 3. Spectral Bandwidth

<badge color="success">Core Feature</badge>

### What is it?

Spectral bandwidth describes how spread out the spectral energy is around the spectral centroid.

Think of:

- Centroid = Centre of a distribution.
- Bandwidth = Spread of that distribution.

### Mathematical Representation

A commonly used spectral bandwidth formula is:

$$
BW_t =
\sqrt{
\frac{
\sum_k |X_t[k]|(f_k-C_t)^2
}{
\sum_k |X_t[k]|
}
}
$$

Where \(C_t\) is the spectral centroid.

### Intuition

- **Narrow bandwidth:** Frequency energy is more concentrated around the centroid.
- **Wide bandwidth:** Frequency energy is more distributed.

### Python Implementation

```python
bandwidth = librosa.feature.spectral_bandwidth(
    y=audio,
    sr=TARGET_SR,
    n_fft=512,
    hop_length=160,
    win_length=400
)[0]
```

### SER Relevance

Bandwidth can help describe differences in spectral distribution between utterances.

But it does not directly measure emotional intensity or vocal harshness. Its interpretation depends on the entire voice and recording context.

---

## 4. Spectral Rolloff

<badge color="success">Core Feature</badge>

### What is it?

Spectral rolloff identifies the frequency below which a specified percentage of the spectral energy is concentrated.

For example, with an 85% rolloff:

Find the frequency below which 85% of the cumulative spectral magnitude lies.

### Mathematical Representation

Find the smallest frequency index \(k_r\) satisfying:

$$
\sum_{k=0}^{k_r}|X_t[k]|^p
\geq
r\sum_{k=0}^{K-1}|X_t[k]|^p
$$

Where:

- \(r\) = Rolloff proportion, such as 0.85.
- \(p=1\) for magnitude-based rolloff.
- \(p=2\) for power-based rolloff.

The rolloff frequency is:

$$
f_{k_r}
$$

### Python Implementation

```python
rolloff = librosa.feature.spectral_rolloff(
    y=audio,
    sr=TARGET_SR,
    n_fft=512,
    hop_length=160,
    win_length=400,
    roll_percent=0.85
)[0]
```

### SER Relevance

Rolloff gives us information about the distribution of higher-frequency spectral content.

It may help distinguish different vocal qualities, but it is not a direct emotion detector.

---

## 5. Spectral Contrast

<badge color="info">Useful Feature</badge>

### What is it?

Spectral contrast measures the difference between spectral peaks and spectral valleys across frequency sub-bands.

In simpler terms:

It describes how strongly the prominent frequency regions stand out compared with the less prominent regions.

Imagine dividing a spectrum into several frequency bands.

Within each band, we estimate:

- A representative high-energy value (peak).
- A representative low-energy value (valley).

Then calculate their difference, often using a logarithmic representation.

### Mathematical Representation

$$
SC_b =
10\log_{10}
\left(
\frac{P_b}{V_b}
\right)
$$

Here:

- \(P_b\) = Estimated peak power in band \(b\).
- \(V_b\) = Estimated valley power in band \(b\).

Implementations can use slightly different estimators.

### Python Implementation

```python
contrast = librosa.feature.spectral_contrast(
    y=audio,
    sr=TARGET_SR,
    n_fft=512,
    hop_length=160,
    n_bands=6
)
```

Notice something different:

Unlike centroid, bandwidth, and rolloff, spectral contrast returns multiple feature sequences, one for each frequency band.

With `n_bands=6`, the output normally contains seven rows: six sub-bands plus the highest-frequency band.

### SER Relevance

It can help describe spectral texture and differences between frequency regions.

It was originally popular in audio classification tasks, but it can also be investigated for speech emotion recognition.

---

## 6. Spectral Flux

<badge color="info">Useful Feature</badge>

This feature is particularly interesting because it captures how the spectrum changes over time.

### What is it?

Spectral flux measures the change between the current frame's spectrum and the previous frame's spectrum.

Imagine:

- Frame 1 has a particular frequency distribution.
- Frame 2 has almost the same distribution.
- Frame 3 has a substantially different distribution.

Frame 3 will have a higher spectral flux.

### Mathematical Representation

One common positive spectral flux definition is:

$$
SF_t =
\sum_k
\left[
\max(0,\hat X_t[k]-\hat X_{t-1}[k])
\right]^2
$$

Where:

- \(\hat X_t[k]\) = Normalized magnitude spectrum at frame \(t\).
- \(\hat X_{t-1}[k]\) = Normalized magnitude spectrum at the previous frame.

This definition measures positive increases in spectral magnitude. Other implementations use different normalization and distance conventions.

### Intuition

- **Lower flux:** Less change between consecutive spectra.
- **Higher flux:** More change between consecutive spectra.

### SER Relevance

It can help characterize changes in vocal activity and spectral dynamics.

For example, a speaker may shift from a calm delivery to a more forceful delivery. Spectral flux may capture some of the accompanying acoustic changes.

But flux also responds to phoneme transitions, consonants, and other ordinary speech changes. It is not emotion-specific.

### Python Implementation

We'll implement a transparent version ourselves in the practical lab.

---

## 7. Spectral Flatness

<badge color="secondary">Optional Feature</badge>

### What is it?

Spectral flatness measures how noise-like or tone-like a spectrum is.

It compares the geometric mean with the arithmetic mean of spectral magnitudes.

### Mathematical Representation

$$
SF =
\frac{
\left(\prod_{k=1}^{K}X[k]\right)^{1/K}
}{
\frac{1}{K}\sum_{k=1}^{K}X[k]
}
$$

In practice, a small epsilon is used to avoid numerical problems.

### General Interpretation

| Spectral Flatness | Interpretation |
|---|---|
| Closer to 0 | More concentrated or tone-like |
| Closer to 1 | More noise-like or flat |

### Python Implementation

```python
flatness = librosa.feature.spectral_flatness(
    y=audio,
    n_fft=512,
    hop_length=160
)[0]
```

### Should We Use It in SER?

It is optional.

Speech contains both harmonic and noise-like components. Flatness may provide additional information, but it can also respond strongly to background noise and recording quality.

We'll extract it for experimentation, but we won't automatically include it in our final model.

---

## 8. Summary of Spectral Features

| Feature | What It Measures | Priority |
|---|---|---|
| Spectral Centroid | Centre of spectral magnitude | Core |
| Spectral Bandwidth | Spread of spectral content | Core |
| Spectral Rolloff | Frequency below which a specified proportion of spectral content is concentrated | Core |
| Spectral Contrast | Differences between spectral peaks and valleys | Useful |
| Spectral Flux | Changes between consecutive spectral frames | Useful |
| Spectral Flatness | How noise-like or tone-like a spectrum is | Optional |

**Important:** These features describe acoustic characteristics. None of them independently identifies an emotion.

---

## 9. Important Research Considerations

### A. Features Can Be Correlated

Some spectral features describe related properties.

For example:

- Spectral centroid and rolloff both describe aspects of frequency distribution.
- Centroid and bandwidth are calculated from the same spectrum.
- Spectral flux measures temporal change rather than simply the average spectral distribution.

Having more features doesn't automatically make a model better.

### B. Speaker Differences Can Dominate Emotion Differences

Imagine:

- Speaker A naturally has a bright, high-frequency voice.
- Speaker B naturally has a darker, lower-frequency voice.

A model might learn speaker identity instead of emotion.

This is why we must eventually evaluate using speaker-independent train/test splits.

### C. Recording Quality Matters

Background noise, microphone distance, compression, and recording equipment can change spectral measurements.

For example, background hiss can increase spectral flatness and affect centroid.

### D. Feature Distributions Are Not Proof of Emotion Recognition

If angry recordings have a higher average centroid in your dataset, that does not prove that anger universally produces a higher centroid.

It is an observation about that dataset.

We need proper validation to determine whether the feature helps classification.

---

## 10. Final Feature Priority for Our SER Project

| Feature | Priority | Decision |
|---|---|---|
| Spectral Centroid | Core | Retain |
| Spectral Bandwidth | Core | Retain |
| Spectral Rolloff | Core | Retain |
| Spectral Contrast | Useful | Retain for experimentation |
| Spectral Flux | Useful | Retain for experimentation |
| Spectral Flatness | Optional | Extract, evaluate, then decide |

For the first classical ML feature set, we'll compare the core features against the expanded feature set rather than automatically including everything.

One more important step: when training a model, fit feature scaling and feature selection using training data only. Never calculate them using the complete dataset before splitting.